# Audience Segmentation & Fan Strategy
Robyn Moore · Last.fm 1K user dataset

Can listening behavior identify audiences worth testing with different artist campaigns? This notebook reproduces the full-cohort workflow in the [case study](case-study.pdf). The published historical results are in the [project README](README.md).

Download the full listening-history ZIP from the [dataset releases](https://github.com/eifuentes/lastfm-dataset-1K/releases) and place it beside this notebook. The 50-user Parquet is an exploratory subset and does not reproduce the full analysis.

> This notebook is published without saved cell outputs because the listener-level archive is not redistributed here. The **Recorded result** notes under each step give the aggregate figures from the original full-cohort run. Running the cells with the full event archive regenerates them.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

HISTORY = Path('userid-timestamp-artid-artname-traid-traname.tsv.zip')
MEMBER = 'userid-timestamp-artid-artname-traid-traname.tsv'
COLUMNS = ['user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name']
SEGMENTS = ['Focused Listeners', 'Exploratory Power Listeners']

def read_history(columns=None):
    if not HISTORY.is_file():
        raise FileNotFoundError(f'Download the full Last.fm listening-history ZIP to {HISTORY}.')
    with ZipFile(HISTORY) as archive, archive.open(MEMBER) as source:
        yield from pd.read_csv(
            source, sep='\t', header=None, names=COLUMNS, chunksize=500_000,
            on_bad_lines='skip', usecols=columns,
        )

## 1. Build listening features
Count the source cohort using the six-field parse, then require at least 500 plays. Follow the original notebook’s projected-column reads for feature aggregation. Observation windows span the first through last observed UTC calendar dates, inclusive. Artist breadth is distinct artist names across that window divided by its length.

In [ ]:
plays = Counter()
events = 0
for chunk in read_history():
    events += len(chunk)
    plays.update(chunk['user_id'].value_counts().to_dict())
eligible = {user for user, count in plays.items() if count >= 500}
print(f'{events:,} parsed events | {len(plays):,} source users | {len(eligible):,} eligible')

artist_counts = Counter()
for chunk in read_history([0, 3]):
    chunk = chunk[chunk['user_id'].isin(eligible)]
    artist_counts.update(chunk.groupby(['user_id', 'artist_name']).size().to_dict())

active_dates = defaultdict(set)
artist_names = defaultdict(set)
for chunk in read_history([0, 1, 3, 5]):
    chunk = chunk[chunk['user_id'].isin(eligible)]
    dates = pd.to_datetime(chunk['timestamp'], errors='coerce', utc=True).dt.normalize()
    for user, days in dates.groupby(chunk['user_id']):
        active_dates[user].update(days.dropna().unique())
    for user, names in chunk.groupby('user_id')['artist_name']:
        artist_names[user].update(names.dropna().unique())

pairs = pd.DataFrame(
    [(user, artist, count) for (user, artist), count in artist_counts.items()],
    columns=['user_id', 'artist_name', 'artist_listens'],
)
top_counts = pairs.groupby('user_id')['artist_listens'].max().rename('top_artist_listens')
features = pd.Series(plays, name='total_listens').rename_axis('user_id').to_frame()
features = features.loc[features.index.isin(eligible)].join(top_counts)
features['active_days'] = [len(active_dates[user]) for user in features.index]
features['window_days'] = [
    (max(active_dates[user]) - min(active_dates[user])).days + 1 if active_dates[user] else np.nan
    for user in features.index
]
features['listens_per_day'] = features['total_listens'] / features['window_days']
features['artist_breadth_per_day'] = pd.Series({user: len(artist_names[user]) for user in features.index}) / features['window_days']
features['top_artist_share_pct'] = 100 * features['top_artist_listens'] / features['total_listens']
features['active_day_rate_pct'] = 100 * features['active_days'] / features['window_days']
METRICS = ['listens_per_day', 'artist_breadth_per_day', 'top_artist_share_pct', 'active_day_rate_pct']
if features[METRICS].isna().any().any():
    raise ValueError('Some eligible listeners have incomplete features. Check the input archive.')

**Recorded result (original full-cohort run).** 19,098,853 parsed events, 992 source users, 926 listeners with at least 500 plays.

## 2. Select and validate the segmentation
Log-transform intensity, breadth, and concentration, then standardize all four inputs. Compare k=2–8 by silhouette. The historical run selected k=2 with a score of 0.3323. Assign the higher-intensity cluster the Exploratory Power label so arbitrary numeric labels do not affect comparisons.

In [ ]:
def standardized(frame):
    values = frame[METRICS].copy()
    values[METRICS[:3]] = np.log1p(values[METRICS[:3]])
    return StandardScaler().fit_transform(values)

def fit(values, k=2, seed=42):
    return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(values)

def segment_names(frame, labels):
    intensity = frame['listens_per_day'].groupby(labels).mean()
    return np.where(labels == intensity.idxmax(), SEGMENTS[1], SEGMENTS[0])

X = standardized(features)
scores = pd.Series({k: silhouette_score(X, fit(X, k)) for k in range(2, 9)}, name='silhouette')
print(scores.round(4).to_string())
labels = fit(X)
features['segment'] = segment_names(features, labels)
profiles = features.groupby('segment')[METRICS].mean().reindex(SEGMENTS)
profiles.insert(0, 'listeners', features['segment'].value_counts())
print(profiles.round(2).to_string())

**Recorded result (original full-cohort run).** Silhouette by number of clusters:

| k | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Silhouette | 0.3323 | 0.2829 | 0.2629 | 0.2592 | 0.2433 | 0.2495 | 0.2431 |

Segment averages at k=2:

| Segment | Listeners | Plays per day | Artist breadth per day | Top-artist share | Active-day rate |
| --- | ---: | ---: | ---: | ---: | ---: |
| Focused Listeners | 454 | 9.67 | 0.62 | 8.67% | 28.22% |
| Exploratory Power Listeners | 472 | 42.19 | 2.49 | 5.92% | 71.36% |

In [ ]:
higher_threshold = features[features['total_listens'] >= 1_000]
higher_labels = segment_names(higher_threshold, fit(standardized(higher_threshold)))
agreement = (higher_threshold['segment'] == higher_labels).mean()
print(f'1,000-play sensitivity: {agreement:.2%} agreement across {len(higher_threshold):,} listeners')
seed_agreement = pd.Series({
    seed: adjusted_rand_score(labels, fit(X, seed=seed))
    for seed in [0, 10, 20, 30, 42, 50, 100]
}, name='adjusted Rand index')
print(seed_agreement.round(4).to_string())

**Recorded result (original full-cohort run).** Raising the minimum to 1,000 plays kept 870 of 890 overlapping listeners in the same segment (97.75%). Different random starts reproduced the same 454 / 472 split.

## 2b. Check breadth at matched play volume
The Exploratory Power segment plays far more music, and distinct artists grow more slowly than plays. Raw artists-per-play therefore favors lighter listeners. To compare like with like, group listeners into play-volume bands and compare median distinct artists within each band. A log-log regression of distinct artists on total plays with a segment term gives one summary figure.

In [ ]:
features['unique_artists'] = pd.Series({user: len(artist_names[user]) for user in features.index})
bands = [500, 2_000, 5_000, 10_000, 20_000, 40_000, 80_000]
features['play_band'] = pd.cut(features['total_listens'], bands, right=False)
matched = features.pivot_table(
    index='play_band', columns='segment', values='unique_artists',
    aggfunc=['count', 'median'], observed=True,
).reindex(columns=SEGMENTS, level=1)
matched['median_ratio'] = matched[('median', SEGMENTS[1])] / matched[('median', SEGMENTS[0])]
print(matched.round(2).to_string())

design = np.column_stack([
    np.ones(len(features)),
    np.log(features['total_listens']),
    features['segment'].eq(SEGMENTS[1]),
]).astype(float)
target = np.log(features['unique_artists'])
coef, *_ = np.linalg.lstsq(design, target, rcond=None)
residual = target - design @ coef
se = np.sqrt(np.diag(residual @ residual / (len(features) - 3) * np.linalg.inv(design.T @ design)))
low, high = np.exp(coef[2] - 1.96 * se[2]), np.exp(coef[2] + 1.96 * se[2])
print(f'Distinct artists at equal play volume: {np.exp(coef[2]):.2f}x for {SEGMENTS[1]} (95% CI {low:.2f}-{high:.2f})')
print(f'Distinct artists scale with plays to the power {coef[1]:.2f}')

**Recorded result (original full-cohort run).** Calculated from the listener-level feature table saved by the original run.

| Total plays | Focused listeners | Exploratory Power listeners | Median artists, Focused | Median artists, Exploratory Power | Ratio |
| --- | ---: | ---: | ---: | ---: | ---: |
| 500–1,999 | 80 | 15 | 241 | 262 | 1.09× |
| 2,000–4,999 | 98 | 37 | 329 | 480 | 1.46× |
| 5,000–9,999 | 109 | 44 | 554 | 698 | 1.26× |
| 10,000–19,999 | 123 | 100 | 583 | 929 | 1.59× |
| 20,000–39,999 | 41 | 145 | 628 | 1,153 | 1.84× |
| 40,000–79,999 | 3 | 103 | 546 | 1,407 | 2.58× |

At equal play volume the Exploratory Power segment heard about 1.38× as many distinct artists (95% CI 1.24–1.53). Distinct artists scaled with plays to the power 0.41. The top band has only three Focused listeners. Artist breadth per day was a clustering input, so part of this gap follows from how the segments were formed.

## 3. Separate artist reach from audience affinity
Penetration is the percentage of a segment with at least one recorded play for an artist. Over-index divides Exploratory Power penetration by Focused penetration. Require at least 20 listeners overall and five in each segment. For the discovery shortlist, also require at least 10% Exploratory Power penetration.

Reach at least once is sensitive to listening volume. This analysis does not isolate genre or artist preference after controlling for how much each segment listens.


In [ ]:
counts = (pairs.join(features['segment'], on='user_id', how='inner')
          .groupby(['artist_name', 'segment']).size().unstack(fill_value=0)
          .reindex(columns=SEGMENTS, fill_value=0))
keep = counts.sum(axis=1).ge(20) & counts.ge(5).all(axis=1)
counts = counts.loc[keep]
penetration = counts.div(profiles['listeners'], axis='columns') * 100
strategy = penetration.rename(columns={name: f'{name} (%)' for name in SEGMENTS})
strategy['reach'] = counts.sum(axis=1)
strategy['over_index'] = penetration[SEGMENTS[1]] / penetration[SEGMENTS[0]]
shortlist = strategy.loc[penetration[SEGMENTS[1]].ge(10)].sort_values('over_index', ascending=False)
print(shortlist.head(10).round(2).to_string())

In [ ]:
examples = ['Radiohead', 'Beach House', 'Flying Lotus', 'Modeselektor']
comparison = penetration.reindex(examples).dropna()
if not comparison.empty:
    ax = comparison.plot.barh(color=['#63839a', '#13775f'], figsize=(8, 3.7))
    ax.set(xlabel='Listeners reached within segment (%)', ylabel='')
    ax.invert_yaxis()
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

**Recorded result (original full-cohort run).** Listener penetration for the four example artists plotted above:

![Artist reach by listener segment for Radiohead, Beach House, Flying Lotus and Modeselektor](artist-affinity.png)

## 4. Turn the results into tests
Radiohead is a broad-reach example. Beach House and Flying Lotus suggest more targeted discovery tests for Exploratory Power Listeners. For Focused Listeners, start with a familiar artist and test whether a catalog or related-artist recommendation increases repeat listening versus a holdout.

The historical cohort is small at the listener level and is not representative of current streaming audiences. Cluster separation is modest, names come from source strings, and artist affinity measures observed association. Demographics were not inputs to the model.